In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from concurrent.futures import ThreadPoolExecutor

# Function to scrape first source
def scrape_stocktitan():
    print("Scraping StockTitan...")
    base_url = "https://www.stocktitan.net"
    url = base_url + "/scanner/momentum"

    columns = ['name', 'change', 'source']
    df = pd.DataFrame(columns=columns)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to retrieve StockTitan page. Status code: {response.status_code}")
        return df

    soup = BeautifulSoup(response.content, "html.parser")
    top_gainers_div = soup.find("div", {"id": 'gainers'})
    top_gainers_table = top_gainers_div.find("div", {"class": 'body'}).find_all("div", {'class': 'content'})

    print(f"StockTitan: Found {len(top_gainers_table)} items.")
    for i in range(min(6, len(top_gainers_table))):
        name_news_div = top_gainers_table[i].find("div", {'class': 'symbol'})
        stock = name_news_div.get_text().split(':')[0].strip() if name_news_div else None
        card_info_divs = top_gainers_table[i].find("div", {'class': "data-group"})
        change_span = card_info_divs.find('span', {'class': 'price-change-ratio'})
        if change_span:
            change = change_span.get_text().replace(r"\r\n", '')
            change_num = float(re.sub(r'[,+%]', '', change))
            print(f"StockTitan - stock: {stock}, change: {change_num}")
            df = pd.concat([df, pd.DataFrame([{'name': stock, 'change': change_num, 'source': 'StockTitan'}])], ignore_index=True)
    return df


# Function to scrape second source
def scrape_stockanalysis():
    print("Scraping StockAnalysis...")
    base_url = "https://stockanalysis.com/"
    url = base_url + "/markets/gainers/"

    columns = ['name', 'change', 'source']
    df = pd.DataFrame(columns=columns)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Failed to retrieve StockAnalysis page. Status code: {response.status_code}")
        return df

    soup = BeautifulSoup(response.content, "html.parser")
    gainers_table = soup.find('table', {'id': 'main-table'})
    table_rows = gainers_table.find_all('tr')
    print(f"StockAnalysis: Found {len(table_rows)} rows.")
    for i in range(1, min(7, len(table_rows))):
        cols_in_row = table_rows[i].find_all('td')
        if len(cols_in_row) >= 4:
            name = cols_in_row[1].get_text()
            change = cols_in_row[3].get_text()
            change_num = float(re.sub(r'[,+%]', '', change))
            print(f"StockAnalysis - stock: {name}, change: {change_num}")
            df = pd.concat([df, pd.DataFrame([{'name': name, 'change': change_num, 'source': 'StockAnalysis'}])], ignore_index=True)
    return df

def get_price_volume_stockanalysis(stock_list: list):
    
    columns = ['name', 'price', 'volume']
    df = pd.DataFrame(columns=columns)

    for stock in stock_list:
        
        # print("stock: ",stock)
        url = f'https://stockanalysis.com/stocks/{stock}/'

        # Send a GET request to the URL
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }
        response = requests.get(url, headers=headers)

        # Check if request was successful
        if response.status_code != 200:
            print(f"Failed to retrieve the page. Status code: {response.status_code}")
            exit()
            new_row = {'name': stock,
                    'price': None,
                    'volume': None}
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        
        else:
            soup = BeautifulSoup(response.content, "html.parser")
            price_table = soup.find('table', {'data-test': 'overview-quote'}).find_all("tr")

            len(price_table)

            price = None
            volume = None
            for row in price_table:
                cols = row.find_all('td')
                if cols[0].get_text() == "Previous Close":
                    price = cols[1].get_text()
                
                elif cols[0].get_text() == "Volume":
                    volume = cols[1].get_text()
            
            new_row = {'name': stock,
                    'price': price,
                    'volume': volume}
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    return df

def get_details_stocktitan(stock_list:list):

    columns = ['name', 'market_cap','float', 'short_percent', 'industry', 'sector', 'country', 'news_link','news_impact_star', 'news_sentiment_star']
    df = pd.DataFrame(columns=columns)

    for stock in stock_list:
    
        url = f"https://www.stocktitan.net/news/{stock}/"
        # Send a GET request to the URL
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }
        response = requests.get(url, headers=headers)

        # Check if request was successful
        if response.status_code != 200:
            print(f"Failed to retrieve the page. Status code: {response.status_code}")
            exit()
            new_row = {
            'name': stock,
            'market_cap': None,
            'float': None,
            'short_percent': None,
            'industry': None,
            'sector': None,
            'country': None,
            'news_link': None,
            'news_impact_star': None,
            'news_sentiment_star': None
            }

            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        
        else:
            # Parse the HTML content using BeautifulSoup
            soup = BeautifulSoup(response.content, "html.parser")
            
            stock_data_div = soup.find_all('div', {'class': 'article-data-panel'})[-1].find_all('div',{'class': 'news-list-item stock-data'})

            market_cap_value = None
            float_value = None
            short_percent = None
            industry = None
            sector = None
            country = None

            for div in stock_data_div:
                # print(div.find('label').get_text())
                # print("==========================")
                if div.find('label').get_text() == "Short Percent":
                    short_percent = div.find('span').get_text()
                
                elif div.find('label').get_text() == "Industry":
                    industry = div.find('span').get_text()

                elif div.find('label').get_text() == "Sector":
                    sector = div.find('span').get_text()

                elif div.find('label').get_text() == "Country":
                    country = div.find('span').get_text()

                elif div.find('label').get_text() == "Market Cap":
                    market_cap_value = div.find('span').get_text()

                elif div.find('label').get_text() == "Float":
                    float_value = div.find('span').get_text()

            news_impact_star = len(soup.find('div', {'class': 'impact-container'}).find_all('div',{'class': 'full'}))
            news_sentiment_star = len(soup.find('div', {'class': 'sentiment-container'}).find_all('div',{'class': 'full'}))

            new_row = {
            'name': stock,
            'market_cap': market_cap_value,
            'float': float_value,
            'short_percent': short_percent,
            'industry': industry,
            'sector': sector,
            'country': country,
            'news_link': url,
            'news_impact_star': news_impact_star,
            'news_sentiment_star': news_sentiment_star
            }

            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    return df

def remove_duplicates_keep_highest(df):
    """
    Removes duplicates based on the 'name' column, keeping the row with the highest 'change' value.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with duplicates removed.
    """
    if 'name' not in df.columns or 'change' not in df.columns:
        raise ValueError("DataFrame must contain 'name' and 'change' columns.")
    
    # Get the index of the row with max 'change' for each 'name'
    idx = df.groupby('name')['change'].idxmax()
    deduped_df = df.loc[idx].reset_index(drop=True)
    
    return deduped_df

def get_top_k_df(df, top_k):
    """
    Returns the top_k rows of the DataFrame based on the 'change' column (descending order).

    Args:
        df (pd.DataFrame): The input DataFrame.
        top_k (int): The number of top rows to return.

    Returns:
        pd.DataFrame: A DataFrame with the top_k rows.
    """
    if 'change' not in df.columns:
        raise ValueError("DataFrame must contain a 'change' column.")
    
    return df.sort_values(by='change', ascending=False).head(top_k).reset_index(drop=True)

def merge_dataframes_on_key(df1, df2, df3, key):
    """
    Merges three DataFrames on the 'name' column using outer joins.

    Args:
        df1 (pd.DataFrame): First DataFrame.
        df2 (pd.DataFrame): Second DataFrame.
        df3 (pd.DataFrame): Third DataFrame.

    Returns:
        pd.DataFrame: A merged DataFrame containing all columns from the three inputs.
    """
    if key not in df1.columns or key not in df2.columns or key not in df3.columns:
        raise ValueError(f"All DataFrames must have a {key} column.")
    
    merged_df = pd.merge(df1, df2, on=key, how='outer')
    merged_df = pd.merge(merged_df, df3, on=key, how='outer')
    
    return merged_df




with ThreadPoolExecutor() as executor:
    future1 = executor.submit(scrape_stocktitan)
    future2 = executor.submit(scrape_stockanalysis)

    df1 = future1.result()
    df2 = future2.result()

# Combine both results
combined_df = pd.concat([df1, df2], ignore_index=True)

# print("\nCombined Results:")
# print(combined_df)


deduped_df = remove_duplicates_keep_highest(combined_df)
# print("\nAfter Removing Duplicates:")
# print(deduped_df)

# Sort the combined dataframe by 'change' in descending order
combined_sorted_df = deduped_df.sort_values(by='change', ascending=False).reset_index(drop=True)
# print("\nSorted Results:")
# print(combined_sorted_df)


top_k_stocks_df = get_top_k_df(combined_sorted_df ,top_k =5)

print("\ntop_k Stocks")
print(top_k_stocks_df)

# with ThreadPoolExecutor() as executor:
#     future1 = executor.submit(scrape_stocktitan)
#     future2 = executor.submit(scrape_stockanalysis)

#     df1 = future1.result()
#     df2 = future2.result()

price_details_df = get_price_volume_stockanalysis(top_k_stocks_df['name'].to_list())

details_df = get_details_stocktitan(top_k_stocks_df['name'].to_list())

print("price_details_df")
print(price_details_df)


print('details_df')
print(details_df)

merged_df = merge_dataframes_on_key(top_k_stocks_df, price_details_df, details_df, key='name')
merged_df.to_csv('merged_df.csv')

Scraping StockTitan...
Scraping StockAnalysis...
StockAnalysis: Found 21 rows.
StockAnalysis - stock: PLRZ, change: 216.89
StockAnalysis - stock: SDM, change: 149.0
StockAnalysis - stock: FRGT, change: 114.26
StockAnalysis - stock: KBSX, change: 96.0
StockAnalysis - stock: IBO, change: 68.25
StockAnalysis - stock: ADIL, change: 48.72


/tmp/ipykernel_925162/3249547084.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{'name': name, 'change': change_num, 'source': 'StockAnalysis'}])], ignore_index=True)


StockTitan: Found 25 items.
StockTitan - stock: PLRZ, change: 219.0
StockTitan - stock: SDM, change: 149.0
StockTitan - stock: FRGT, change: 115.0
StockTitan - stock: KBSX, change: 96.0
StockTitan - stock: IBO, change: 67.0
StockTitan - stock: ADIL, change: 45.0

top_k Stocks
   name  change         source
0  PLRZ  219.00     StockTitan
1   SDM  149.00     StockTitan
2  FRGT  115.00     StockTitan
3  KBSX   96.00     StockTitan
4   IBO   68.25  StockAnalysis


/tmp/ipykernel_925162/3249547084.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{'name': stock, 'change': change_num, 'source': 'StockTitan'}])], ignore_index=True)


Failed to retrieve the page. Status code: 404
price_details_df
   name   price       volume
0  PLRZ  0.2534  194,087,854
1   SDM    None      389,316
2  FRGT   0.985  125,853,422
3  KBSX   1.500      424,664
4   IBO  0.4000   86,235,072
details_df
   name market_cap   float short_percent                industry  \
0  PLRZ     13.14M   4.57M          1.3%           Biotechnology   
1   SDM       None    None          None                    None   
2  FRGT      2.21M   2.26M         1.84%  Software - Application   
3  KBSX     88.64M  43.71M         0.01%                   Steel   
4   IBO      5.15M   7.34M         0.08%           Biotechnology   

                                             sector        country  \
0                                        Healthcare         Israel   
1                                              None           None   
2  Arrangement of Transportation of Freight & Cargo  United States   
3                                   Basic Materials         Tai

: 

In [30]:
combined_sorted_df.to_csv('combined_df.csv',index=False)
top_gainers_combined.to_csv('top_gainers_combined.csv', index=False)